In [4]:
import pandas as pd
import numpy as np

StatementMeta(, 55e1f0d2-1aaa-465a-acd2-19dd2488d4c8, 6, Finished, Available, Finished)

In [5]:
# Load the orders data in a pandas dataframe
orders = pd.read_csv("/lakehouse/default/Files/olist_orders_dataset.csv")
print(orders.head())

StatementMeta(, 55e1f0d2-1aaa-465a-acd2-19dd2488d4c8, 7, Finished, Available, Finished)

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08

In [6]:
profile = pd.DataFrame({
    'Column': orders.columns.values,
    'negative(%)': [
        len(orders[col][orders[col] < 0]) / len(orders) * 100 if col in orders.select_dtypes(include=[np.number]).columns else 0
        for col in orders.columns
    ],  
    'zero(%)': [
        len(orders[col][orders[col] == 0]) / len(orders) * 100 if col in orders.select_dtypes(include=[np.number]).columns else 0
        for col in orders.columns
    ],  
    'duplicates': orders.duplicated().sum(), 
    'unique': orders.nunique().values, 
})

profile

StatementMeta(, 55e1f0d2-1aaa-465a-acd2-19dd2488d4c8, 8, Finished, Available, Finished)

,Column,negative(%),zero(%),duplicates,unique
0,order_id,0,0,0,99441
1,customer_id,0,0,0,99441
2,order_status,0,0,0,8
3,order_purchase_timestamp,0,0,0,98875
4,order_approved_at,0,0,0,90733
5,order_delivered_carrier_date,0,0,0,81018
6,order_delivered_customer_date,0,0,0,95664
7,order_estimated_delivery_date,0,0,0,459


In [7]:
# Insert the data cleaning steps / functions here
# There are some null values in the order timestamp fields that represent less than 3% of total nulls.
# Most are reflected under order_status='canceled', 'processing', 'shipped', 'invoiced', but there are some 
# reflected as 'delivered'.
# We will filter out those orders where 'order_status'=delivered and simultaneously there are null values for 'order_delivered_customer_date'

from pyspark.sql.functions import col, isnull, when
from pyspark.sql.types import TimestampType
from datetime import date, timedelta
from pyspark.sql.functions import to_timestamp
from pyspark.sql.functions import regexp_replace, trim
from pyspark.sql.types import StringType


orders_spark = spark.createDataFrame(orders)


orders_cleaned = orders_spark.filter(
    ~(
        (col('order_status') == 'delivered') & 
        (col('order_delivered_customer_date').isNull())
    )
)


# Convert order time related fields to timestamp format

timestamp_columns_to_convert = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", 
"order_delivered_customer_date", "order_estimated_delivery_date"] 

orders_cleaned = orders_cleaned.select(
    *[to_timestamp(col(c), "yyyy-MM-dd HH:mm:ss").alias(c) if c in timestamp_columns_to_convert else col(c) for c in orders_cleaned.columns]
)

for column in orders_cleaned.columns:
    # Cast to string
    orders_cleaned = orders_cleaned.withColumn(column, col(column).cast(StringType()))
    # Replace non-breaking space
    orders_cleaned = orders_cleaned.withColumn(column, regexp_replace(col(column), '\u00a0', ' '))
    # Replace control characters
    orders_cleaned = orders_cleaned.withColumn(column, regexp_replace(col(column), '[\x00-\x1f\x7f-\x9f]', ''))
    # Replace carriage return
    orders_cleaned = orders_cleaned.withColumn(column, regexp_replace(col(column), '\r', ' '))
    # Replace newline
    orders_cleaned = orders_cleaned.withColumn(column, regexp_replace(col(column), '\n', ' '))
    # Strip spaces from start/end
    orders_cleaned = orders_cleaned.withColumn(column, trim(col(column)))


display(orders_cleaned)

StatementMeta(, 55e1f0d2-1aaa-465a-acd2-19dd2488d4c8, 9, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 1940b7f2-c823-4347-a533-e7d42659bcfd)

In [9]:
# Check dataframe post-cleaning

# Check no nulls

assert orders_cleaned.filter(orders_cleaned['order_id'].isNull()).count() == 0, "Null values found in order id"
assert orders_cleaned.filter(orders_cleaned['customer_id'].isNull()).count() == 0, "Null values found in order id"
assert orders_cleaned.filter(orders_cleaned['order_status'].isNull()).count() == 0, "Null values found in order id"

# Check timestamp format

timestamp_columns = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", 
"order_delivered_customer_date", "order_estimated_delivery_date"] 

for col_name in timestamp_columns:
    # Extract the data type of the column from the schema
    col_type = [field.dataType for field in orders_cleaned.schema.fields if field.name == col_name][0]
    
    # Assert the column is of TimestampType
    assert isinstance(col_type, TimestampType), f"Column '{col_name}' is not datetime but {col_type}"



StatementMeta(, 55e1f0d2-1aaa-465a-acd2-19dd2488d4c8, 11, Finished, Available, Finished)

In [5]:
# Write the table to the silver lakehouse as a delta table
# Save as a Delta table in Silver Lakehouse

silver_path = "SilverLakehouse.dbo.olist_orders_cleaned"
orders_cleaned.write.format("delta").mode("overwrite").saveAsTable(silver_path)

StatementMeta(, c52d3bf6-352f-4239-bdc9-d34a4b7524b1, 7, Finished, Available, Finished)